In [1]:
import pandas as pd
import numpy as np
from surprise import SVD, Dataset, Reader
from surprise.model_selection import cross_validate
from surprise import accuracy

train_df = pd.read_csv('../data/processed/train.csv')
test_df  = pd.read_csv('../data/processed/test.csv')

reader = Reader(rating_scale=(1, 5))
train_data = Dataset.load_from_df(
    train_df[['userId', 'movieId', 'rating']], reader
)

print(f"Train: {len(train_df):,}개")
print(f"Test:  {len(test_df):,}개")
print("데이터 로드 완료")

Train: 80,000개
Test:  20,000개
데이터 로드 완료


In [2]:
svd = SVD(n_factors=100, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=42)

cv_results = cross_validate(svd, train_data, measures=['RMSE', 'MAE'],
                             cv=5, verbose=True)

rmse_mean = cv_results['test_rmse'].mean()
mae_mean  = cv_results['test_mae'].mean()

print(f"\n5-Fold 평균 RMSE: {rmse_mean:.4f}")
print(f"5-Fold 평균 MAE:  {mae_mean:.4f}")
print(f"\n목표(벤치마크): RMSE ≤ 0.9364")

if rmse_mean <= 0.9364:
    print("✓ 베이스라인 목표 달성 — 2주차 진행 가능")
else:
    print("✗ 하이퍼파라미터 조정 필요")

Evaluating RMSE, MAE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.9373  0.9381  0.9288  0.9392  0.9340  0.9355  0.0037  
MAE (testset)     0.7404  0.7390  0.7319  0.7403  0.7334  0.7370  0.0036  
Fit time          0.62    0.58    0.62    0.59    0.62    0.61    0.02    
Test time         0.09    0.06    0.09    0.06    0.10    0.08    0.02    

5-Fold 평균 RMSE: 0.9355
5-Fold 평균 MAE:  0.7370

목표(벤치마크): RMSE ≤ 0.9364
✓ 베이스라인 목표 달성 — 2주차 진행 가능


In [3]:
# 전체 train으로 학습 후 test로 평가
trainset_full = train_data.build_full_trainset()
svd.fit(trainset_full)

testset = [
    (row['userId'], row['movieId'], row['rating'])
    for _, row in test_df.iterrows()
]
predictions = svd.test(testset)

def precision_recall_ndcg_at_k(predictions, k=10, threshold=3.5):
    from collections import defaultdict
    user_est_true = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        user_est_true[uid].append((est, true_r))

    precisions, recalls, ndcgs = {}, {}, {}
    for uid, user_ratings in user_est_true.items():
        user_ratings.sort(key=lambda x: x[0], reverse=True)
        top_k   = user_ratings[:k]
        n_rel   = sum(1 for (_, r) in user_ratings if r >= threshold)
        n_hit   = sum(1 for (_, r) in top_k if r >= threshold)

        precisions[uid] = n_hit / k
        recalls[uid]    = n_hit / n_rel if n_rel > 0 else 0

        dcg  = sum((2**(r >= threshold) - 1) / np.log2(i + 2)
                   for i, (_, r) in enumerate(top_k))
        idcg = sum(1 / np.log2(i + 2) for i in range(min(n_rel, k)))
        ndcgs[uid] = dcg / idcg if idcg > 0 else 0

    return (sum(precisions.values()) / len(precisions),
            sum(recalls.values())    / len(recalls),
            sum(ndcgs.values())      / len(ndcgs))

prec, rec, ndcg = precision_recall_ndcg_at_k(predictions, k=10, threshold=3.5)
rmse = accuracy.rmse(predictions, verbose=False)
mae  = accuracy.mae(predictions,  verbose=False)

print(f"RMSE:         {rmse:.4f}")
print(f"MAE:          {mae:.4f}")
print(f"Precision@10: {prec:.4f}")
print(f"Recall@10:    {rec:.4f}")
print(f"NDCG@10:      {ndcg:.4f}")

RMSE:         1.0300
MAE:          0.8214
Precision@10: 0.6518
Recall@10:    0.4262
NDCG@10:      0.7823


In [4]:
import os
os.makedirs('../results', exist_ok=True)

baseline = pd.DataFrame([{
    'model':        'SVD_baseline',
    'RMSE':         round(rmse, 4),
    'MAE':          round(mae, 4),
    'Precision@10': round(prec, 4),
    'Recall@10':    round(rec, 4),
    'NDCG@10':      round(ndcg, 4),
    'ILD':          None
}])

baseline.to_csv('../results/baseline_metrics.csv', index=False)
print("저장 완료")
print(baseline.to_string(index=False))

저장 완료
       model  RMSE    MAE  Precision@10  Recall@10  NDCG@10  ILD
SVD_baseline  1.03 0.8214        0.6518     0.4262   0.7823 None
